## Bronze to Silver: Clean Raw CSVs

**Input:** Raw CSVs from Blob Storage

**Output:** Bronze Delta tables (cleaned)


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Bronze to Silver — Clean Raw CSVs

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import LongType

storage_key = dbutils.secrets.get(scope="retailrocket", key="storage-key")
spark.conf.set("fs.azure.account.key.stdportfolio.blob.core.windows.net", storage_key)

# COMMAND ----------

# Create schemas
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 1. Load Raw CSVs → Bronze

# COMMAND ----------

events_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("wasbs://raw@stdportfolio.blob.core.windows.net/data-retailrocket/events.csv")
)

category_tree_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("wasbs://raw@stdportfolio.blob.core.windows.net/data-retailrocket/category_tree.csv")
)

item_properties_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("wasbs://raw@stdportfolio.blob.core.windows.net/data-retailrocket/item_properties_part*.csv")
)

print(f"Events:          {events_raw.count():,} rows")
print(f"Category tree:   {category_tree_raw.count():,} rows")
print(f"Item properties: {item_properties_raw.count():,} rows")

# COMMAND ----------

# Clean events for bronze
events_clean = (
    events_raw
    .withColumn("timestamp_ms", F.col("timestamp").cast(LongType()))
    .withColumn("event_timestamp", F.to_timestamp(F.col("timestamp_ms") / 1000))
    .withColumn("visitorid", F.col("visitorid").cast("long"))
    .withColumn("itemid", F.col("itemid").cast("long"))
    .withColumn("transactionid", F.col("transactionid").cast("long"))
    .filter(F.col("visitorid").isNotNull())
    .filter(F.col("itemid").isNotNull())
)

# Write bronze tables
events_clean.write.format("delta").mode("overwrite").saveAsTable("bronze.events")
category_tree_raw.write.format("delta").mode("overwrite").saveAsTable("bronze.category_tree")
item_properties_raw.write.format("delta").mode("overwrite").saveAsTable("bronze.item_properties")

print(f"Bronze events: {events_clean.count():,} rows")
print("Bronze tables written.")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2. Bronze → Silver — Deduplicate & Clean

# COMMAND ----------

events = spark.table("bronze.events")
category_tree = spark.table("bronze.category_tree")
item_properties = spark.table("bronze.item_properties")

# COMMAND ----------

# Silver: events — remove null timestamps, deduplicate
silver_events = (
    events
    .filter(F.col("timestamp_ms").isNotNull())
    .dropDuplicates(["timestamp_ms", "visitorid", "itemid", "event"])
)

# Silver: category_tree — no changes needed
silver_category_tree = category_tree

# Silver: item_properties — keep only readable properties and remove hashed values, deduplicate
silver_item_properties = (
    item_properties
    .withColumn("timestamp_ms", F.col("timestamp").cast(LongType()))
    .filter(F.col("property").isin("categoryid", "available"))
    .dropDuplicates(["itemid", "property", "timestamp_ms"])
)

# COMMAND ----------

# Write silver tables
silver_events.write.format("delta").mode("overwrite").saveAsTable("silver.events")
silver_category_tree.write.format("delta").mode("overwrite").saveAsTable("silver.category_tree")
silver_item_properties.write.format("delta").mode("overwrite").saveAsTable("silver.item_properties")

print(f"Silver events:          {silver_events.count():,} rows")
print(f"Silver category_tree:   {silver_category_tree.count():,} rows")
print(f"Silver item_properties: {silver_item_properties.count():,} rows")
print("Silver tables written.")